# 1 load package and data`

In [ ]:
import json
from collections import namedtuple, Counter

import numpy as np
import pandas as pd
import scanpy as sc
import pandas as pd
import squidpy as sq

## Image manipulation and geometry
from tifffile import imread
from skimage.io import imread as skimread

## Plotting imports
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, Colormap
from matplotlib.cm import ScalarMappable
from matplotlib.colorbar import ColorbarBase
from matplotlib.lines import Line2D
from matplotlib import rc_context

In [ ]:
folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\embryo"
h5ad_file = Path(folder_path) / "whole_embryo_hvg_res_1_updata.h5ad"
embryo = sc.read_h5ad(h5ad_file)

# 2 process selected cluster of Cardiac population

In [ ]:
import numpy as np
keep = np.array(["0","12","20","25"], dtype=str)

nt = adata[adata.obs["leiden_hvg_1"].isin(keep)].copy()

In [ ]:
import scanpy as sc


if nt.n_obs == 0:
    print("Warning: nt is empty. Aborting processing.")
else:
    print(f"Info: nt selected with {nt.n_obs} cells.")
    

    if 'counts_raw' in nt.layers:
        nt.X = nt.layers['counts_raw'].copy()
        print("Info: .X initialized from 'counts_raw' layer.")
    

    sc.pp.normalize_total(nt, inplace=True)
    sc.pp.log1p(nt)
    

    sc.pp.highly_variable_genes(nt, n_top_genes=2000)
    

    nt.raw = nt.copy()
    

    sc.pp.scale(nt, max_value=10)
    

    sc.tl.pca(nt, use_highly_variable=True)
    

    sc.pp.neighbors(nt)
    

    sc.tl.umap(nt, min_dist=0.01, spread=2)
    

    sc.tl.leiden(nt, resolution=0.5, key_added='leiden_sub')
    
    print(f"nt (n={nt.n_obs}) processing complete.")
    print(f"Total genes retained: {nt.n_vars}")
    print(f"Highly variable genes used for analysis: {nt.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(nt, 
                        groupby="leiden_hvg_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(nt, group=None)
deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure2\DEG\DEG_leiden_hvg_sub.xlsx", index=True)

In [ ]:
sc.tl.dendrogram(nt, groupby="leiden_hvg_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',
sc.pl.dendrogram(nt, groupby="leiden_hvg_sub")

In [ ]:
nt.write_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\embryont_nmp_nc_2026_02_06.h5ad")

# 3 Figure 3a

In [ ]:
sub_cluster_palette = {

    "e0_0": "#ff002a",  
    "e0_1": "#000080",  
    "e0_2": "#e5e022",  
    "e0_3": "#009203",  
    "e0_4": "#a8329e",  
    "e0_5": "#00FFFF",  

  
    "e12_0": "#FF8C00", 
    "e12_1": "#565DFD", 
    "e12_2": "#FF00FF", 
    "e12_3": "#8B4513", 

 
    "e20_0": "#66c102", 
    "e20_1": "#8e0119", 
    "e20_2": "#10686f", 

-
    "e25_0": "#FF1493", 
    "e25_1": "#9f50f9", 
    "e25_2": "#32a88e" , 
}

In [ ]:

tsne_coords =nt.obsm['X_umap']


leiden_clusters = nt.obs['leiden_hvg_sub']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['tSNE_1', 'tSNE_2'])
tsne_df['leiden_hvg_sub'] = leiden_clusters.values

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='tSNE_1', y='tSNE_2', hue='leiden_hvg_sub', palette=sub_cluster_palette, s=5, edgecolor=None)
plt.title("t-SNE Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure2\umap_heatmap\umap_nt_hvg_sub.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

# 4 Figure 2b

In [ ]:
my_genes=[
     "CDX2","PDPN", "WNT3A","SHH","CGNL1", "ITM2A","FOXA1" ,"OTX2","EN1", "NKX2-1", "FEZF1", "HESX1",
    "IRX1","ILDR2", "NTN1", "PAX3", "DLL3"
]
sc.pl.dotplot(
    nt,
    var_names=my_genes,
    groupby='leiden_hvg_sub',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(8, 4),
    dendrogram=True,
    save='NT_NMP_NT_leiden_sub_normalize.pdf'
)

# 5 Figure 2e, f

In [ ]:
import pandas as pd


file_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\From Jamilla\DASH_APP\11_2026_02_05_for_NT\coordinate_mapping_v4.2.csv"

coord_mapping = pd.read_csv(file_path)

In [ ]:
coord_mapping["cell_id_1"]=coord_mapping["id"]
coord_mapping_selected = coord_mapping[['cell_id_1', 'new_x', 'new_y', 'new_z']].rename(columns={
    'new_x': 'DV',
    'new_y': 'LR',
    'new_z': 'AP'
})

nt.obs = nt.obs.merge(coord_mapping_selected, on='cell_id_1', how='left')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.interpolate import make_interp_spline
from scipy.stats import gaussian_kde
from scipy.ndimage import gaussian_filter1d

def _intersections_on_grid(x, y1, y2):
    """ x """
    d = y1 - y2
    s = np.sign(d)
    idx = np.where(np.diff(s) != 0)[0]    # 
    xs = []
    for i in idx:
        x0, x1 = x[i], x[i+1]
        d0, d1 = d[i], d[i+1]
        if np.isfinite(d0) and np.isfinite(d1) and (d1 != d0):
            xs.append(x0 - d0 * (x1 - x0) / (d1 - d0))
    return xs

def plot_binned_density(
    adata,
    coord_col='AP',                 #  ( 'AP', 'DV', 'LR')
    group_col='leiden_1',           #  ( 'leiden_1', 'cell_type')
    n_bins=50,
    palette=None,
    figsize=(15, 6),
    xlim=None,
    ylim=None,
    fill=True,
    fill_alpha=0.5,                 # 0.5
    linewidth=2.5,
    smooth=0,
    show_legend=True,
    vertical_lines=None,
    vline_color='blue',
    vline_style='--',
    vline_width=1.5,
    vline_alpha=0.8,
    density_method='pdf',
    pairs_to_draw=None,
    post_smooth_sigma=0,
    dashed_groups=None,             #  ['1', '3', '5']
    secondary_y_groups=None,        # Y ['10', '11']
    secondary_ylim=None,            # Y
    x_tick_interval=100,            # X100
    primary_y_groups=None,          # 🆕 Y ['0', '1', '2']None
):
    """
     (coord_col)  bins  (group_col) 
    
    :
    - fill_alpha: float,  (0-1)0.5
    - dashed_groups: list of str, 
    - secondary_y_groups: list of str, Y
    - secondary_ylim: tuple, Y (ymin, ymax)
    - x_tick_interval: int or float, X100
    - primary_y_groups: list of str, YNonesecondary
    """

    # —— 1) 
    plot_df = adata.obs.copy()
    
    # 
    if coord_col not in plot_df.columns:
        raise KeyError(f"obs  '{coord_col}'")
    if group_col not in plot_df.columns:
        raise KeyError(f"obs  '{group_col}'")
    
    # 
    plot_df[coord_col] = pd.to_numeric(plot_df[coord_col], errors='coerce')
    plot_df = plot_df.dropna(subset=[coord_col, group_col])
    
    if plot_df.empty:
        print("Warning: No valid data found.")
        return None, {}

    #  cluster  1 vs '1' 
    plot_df[group_col] = plot_df[group_col].astype(str)
    all_groups = sorted(plot_df[group_col].unique(), key=lambda x: (len(x), x))

    # 
    dashed_groups = [str(g) for g in dashed_groups] if dashed_groups else []
    secondary_y_groups = [str(g) for g in secondary_y_groups] if secondary_y_groups else []
    primary_y_groups = [str(g) for g in primary_y_groups] if primary_y_groups else None

    # 🆕 
    if primary_y_groups is None and secondary_y_groups:
        # secondarysecondary
        groups_to_plot = all_groups
    elif primary_y_groups is not None:
        # primary_y_groupsprimary + secondary
        groups_to_plot = list(set(primary_y_groups) | set(secondary_y_groups))
        # 
        groups_to_plot = [g for g in all_groups if g in groups_to_plot]
    else:
        # 
        groups_to_plot = all_groups

    # —— 2)  bin
    coord_min, coord_max = plot_df[coord_col].min(), plot_df[coord_col].max()
    bins = np.linspace(coord_min, coord_max, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_width = bins[1] - bins[0]

    # —— 3)  group //
    densities = {}
    for cl in groups_to_plot:  # 🆕 
        #  group_col  coord_col
        vals = plot_df.loc[plot_df[group_col] == cl, coord_col].to_numpy()

        if density_method == 'kde':
            try:
                kde = gaussian_kde(vals)
                density = kde(bin_centers)
            except Exception:
                # 
                hist, _ = np.histogram(vals, bins=bins)
                density = hist / (len(vals) * bin_width if len(vals) > 0 else 1)
        else:
            hist, _ = np.histogram(vals, bins=bins)
            if density_method == 'pdf':
                density = hist / (len(vals) * bin_width if len(vals) > 0 else 1)
            elif density_method == 'frequency':
                density = hist / bin_width
            elif density_method == 'relative':
                density = hist / (len(vals) if len(vals) > 0 else 1)
            elif density_method == 'count':
                density = hist.astype(float)
            elif density_method == 'normalized':
                m = hist.max() if hist.size > 0 else 0
                density = hist / (m + 1e-10)
            else:
                raise ValueError(f": {density_method}")

        densities[cl] = density

    # —— 3.5)  + 
    x_dense = np.linspace(bin_centers.min(), bin_centers.max(), n_bins * 20)
    y_dense = {}
    for cl in groups_to_plot:  # 🆕 
        y = densities[cl]
        y_interp = np.interp(x_dense, bin_centers, y)
        if post_smooth_sigma and post_smooth_sigma > 0 and density_method != 'kde':
            y_interp = gaussian_filter1d(y_interp, sigma=post_smooth_sigma)
        y_dense[cl] = y_interp

    # —— 3.6) 
    intersections = {}  # {(cl1, cl2): [x1, x2, ...]}
    for i, c1 in enumerate(groups_to_plot):
        for j in range(i+1, len(groups_to_plot)):
            c2 = groups_to_plot[j]
            xs = _intersections_on_grid(x_dense, y_dense[c1], y_dense[c2])
            intersections[(c1, c2)] = xs

    # —— 4)  - Y
    fig, ax = plt.subplots(figsize=figsize)
    
    # Y
    ax2 = None
    if secondary_y_groups:
        ax2 = ax.twinx()

    # 🆕 
    if primary_y_groups is not None:
        # primary_y_groups
        primary_groups = [g for g in groups_to_plot if g in primary_y_groups and g not in secondary_y_groups]
    else:
        # secondary
        primary_groups = [g for g in groups_to_plot if g not in secondary_y_groups]
    
    secondary_groups = [g for g in groups_to_plot if g in secondary_y_groups]

    # Y
    for cl in primary_groups:
        color = palette.get(cl, None) if palette else None
        y = densities[cl]
        
        # 
        linestyle = '--' if cl in dashed_groups else '-'

        if smooth > 0 and density_method != 'kde':
            try:
                x_smooth = np.linspace(bin_centers.min(), bin_centers.max(), n_bins * 10)
                k = min(int(smooth), len(bin_centers) - 1, 5)
                spl = make_interp_spline(bin_centers, y, k=k)
                y_s = np.maximum(spl(x_smooth), 0)
                if fill:
                    ax.fill_between(x_smooth, y_s, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                ax.plot(x_smooth, y_s, linewidth=linewidth, color=color,
                       linestyle=linestyle,
                       label=f'{group_col} {cl}' if not fill else '')
            except Exception:
                if fill:
                    ax.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                ax.plot(bin_centers, y, linewidth=linewidth, color=color,
                       linestyle=linestyle,
                       label=f'{group_col} {cl}' if not fill else '')
        else:
            if fill:
                ax.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
            ax.plot(bin_centers, y, linewidth=linewidth, color=color,
                   linestyle=linestyle,
                   label=f'{group_col} {cl}' if not fill else '')

    # Y
    if ax2:
        for cl in secondary_groups:
            color = palette.get(cl, None) if palette else None
            y = densities[cl]
            
            linestyle = '--' if cl in dashed_groups else '-'

            if smooth > 0 and density_method != 'kde':
                try:
                    x_smooth = np.linspace(bin_centers.min(), bin_centers.max(), n_bins * 10)
                    k = min(int(smooth), len(bin_centers) - 1, 5)
                    spl = make_interp_spline(bin_centers, y, k=k)
                    y_s = np.maximum(spl(x_smooth), 0)
                    if fill:
                        ax2.fill_between(x_smooth, y_s, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                    ax2.plot(x_smooth, y_s, linewidth=linewidth, color=color,
                            linestyle=linestyle,
                            label=f'{group_col} {cl}' if not fill else '')
                except Exception:
                    if fill:
                        ax2.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                    ax2.plot(bin_centers, y, linewidth=linewidth, color=color,
                            linestyle=linestyle,
                            label=f'{group_col} {cl}' if not fill else '')
            else:
                if fill:
                    ax2.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                ax2.plot(bin_centers, y, linewidth=linewidth, color=color,
                        linestyle=linestyle,
                        label=f'{group_col} {cl}' if not fill else '')

    # 
    if vertical_lines is not None:
        for xv in vertical_lines:
            ax.axvline(xv, color=vline_color, linestyle=vline_style,
                        linewidth=vline_width, alpha=vline_alpha, zorder=10)

    #  pair 
    if pairs_to_draw:
        for p in pairs_to_draw:
            c1, c2 = map(str, p)  # 
            xs = intersections.get((c1, c2), []) + intersections.get((c2, c1), [])
            for xv in xs:
                ax.axvline(xv, color='k', linestyle=':', lw=1.2, alpha=0.9, zorder=11)

    # 
    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)
    if secondary_ylim and ax2: ax2.set_ylim(secondary_ylim)
    
    ax.set_xlabel(f"{coord_col} Position", fontsize=12)

    ylabel_dict = {
        'pdf': 'Probability Density',
        'frequency': 'Frequency Density',
        'relative': 'Relative Frequency',
        'count': 'Cell Count',
        'kde': 'Density (KDE)',
        'normalized': 'Normalized Density'
    }
    ax.set_ylabel(ylabel_dict.get(density_method, 'Density'), fontsize=12)
    
    # Y
    if ax2:
        ax2.set_ylabel(f'{ylabel_dict.get(density_method, "Density")} (Secondary)', 
                       fontsize=12)
    
    ax.set_title(f'{coord_col} Distribution of {group_col} Clusters ({density_method.upper()} Method)',
                 fontsize=18, fontweight='bold', pad=20)

    ax.xaxis.set_major_locator(mticker.MultipleLocator(x_tick_interval))
    ax.tick_params(axis='x', rotation=90, labelsize=10)
    
    #  - Y
    if show_legend:
        lines1, labels1 = ax.get_legend_handles_labels()
        if ax2:
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, 
                     bbox_to_anchor=(1.15, 1), loc='upper left')
        else:
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    ax.margins(x=0, y=0.02)
    plt.tight_layout()

    # 
    info = {
        "bin_centers": bin_centers,
        "densities": densities,      #  bin 
        "x_dense": x_dense,
        "y_dense": y_dense,          # 
        "intersections": intersections,
        "ax2": ax2,                  # 
        "primary_groups": primary_groups,  # 🆕 
        "secondary_groups": secondary_groups  # 🆕 
    }
    return fig, info

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import matplotlib.ticker as mticker

# ============================================================
# Fully enhanced version: Plot gene expression along AP axis (supports subset filtering and dual Y-axes)
# ============================================================
def plot_genes_along_AP(
    adata,                          # NEW: AnnData object
    gene_list=None,                 # Gene name list (simple mode)
    gene_configs=None,              # NEW: Gene configuration list (advanced mode)
    coord_col='AP',                 # NEW: coordinate column name
    subset_col='sub_leiden',        # NEW: column used for subset filtering
    n_bins=50, 
    show_points=True, 
    smooth=0, 
    figsize=(12, 6), 
    linewidth=2, 
    fill=False,
    fill_alpha=0.3,                 # NEW: fill transparency
    gene_colors=None, 
    gene_linestyles=None,
    vertical_lines=None, 
    vline_color='blue',
    vline_style='--', 
    vline_width=1.5, 
    vline_alpha=0.8,
    show_legend=True,
    xlim=None,                      # NEW: X-axis limits, e.g. (0, 2500)
    ylim=None,                      # NEW: Primary (left) Y-axis limits, e.g. (0,1)
    x_tick_interval=100,            # NEW: X-axis tick interval
    secondary_ylim=None,            # NEW: Secondary (right) Y-axis limits
    normalization='minmax'          # NEW: normalization method: 'minmax', 'zscore', 'none'
):
    

    if gene_configs is None and gene_list is None:
        raise ValueError("Either gene_list or gene_configs must be provided")
    
    # 检查列是否存在
    if coord_col not in adata.obs.columns:
        raise KeyError(f"Coordinate column not found in obs '{coord_col}'")
    if subset_col not in adata.obs.columns:
        raise KeyError(f"Subset column not found in obs '{subset_col}'")
    
    # 简单模式：Convert gene_list to gene_configs format
    if gene_configs is None:
        gene_configs = [{
            'genes': gene_list,
            'subset_values': None,  # Use all cells
            'use_secondary_y': False,
            'linestyle': None,  # use gene_linestyles
            'color': None       # use gene_colors
        }]
    
    # ========== 2. Create figure and axes ==========
    fig, ax = plt.subplots(figsize=figsize)
    
    # Check whether a secondary Y-axis is needed
    need_secondary_y = any(config.get('use_secondary_y', False) for config in gene_configs)
    ax2 = None
    if need_secondary_y:
        ax2 = ax.twinx()
    
    # ========== 3. Process each configuration group ==========
    all_gene_curves = {}
    
    for config in gene_configs:
        genes = config['genes']
        subset_values = config.get('subset_values', None)
        use_secondary_y = config.get('use_secondary_y', False)
        config_linestyle = config.get('linestyle', '-')
        config_color = config.get('color', None)
        config_n_bins = config.get('n_bins', n_bins)  # Using 配置的n_bins或全局n_bins
        
        # Select current Y-axis
        current_ax = ax2 if (use_secondary_y and ax2 is not None) else ax
        
        # Filter cell subset
        if subset_values is not None:
            # Convert subset_values to string list
            subset_values_str = [str(v) for v in subset_values]
            mask = adata.obs[subset_col].astype(str).isin(subset_values_str)
            adata_subset = adata[mask, :].copy()
            print(f"Using {subset_col}中值为{subset_values} cells, total {mask.sum()} cells, bins={config_n_bins}")
        else:
            adata_subset = adata
            print(f"Use all cells，共{adata_subset.n_obs} cells, bins={config_n_bins}")
        
        # Get and sort coordinate values
        coords = adata_subset.obs[coord_col].values
        order = np.argsort(coords)
        coords_sorted = coords[order]
  
        bins = np.linspace(coords_sorted.min(), coords_sorted.max(), config_n_bins+1)
        bin_indices = np.digitize(coords_sorted, bins) - 1
        bin_indices = np.clip(bin_indices, 0, config_n_bins-1)
        
        # Calculate bin centers
        bin_centers = (bins[:-1] + bins[1:]) / 2
        
        # ========== 4. Calculate expression curve for each gene ==========
        for gene in genes:
            if gene not in adata_subset.var_names:
                print(f"Warning: {gene}  not found, skipped")
                continue
            
            # Get expression
            expr = adata_subset[:, gene].X
            if hasattr(expr, 'toarray'):
                expr = expr.toarray().ravel()
            else:
                expr = expr.ravel()
            expr_sorted = expr[order]
            
            # Calculate mean expression for each bin
            bin_means = np.array([expr_sorted[bin_indices == i].mean() 
                                  if np.any(bin_indices == i) else 0 
                                  for i in range(config_n_bins)])  # Using config_n_bins
            
            # Normalization
            if normalization == 'minmax':
                bin_means_norm = (bin_means - bin_means.min()) / (bin_means.max() - bin_means.min() + 1e-10)
            elif normalization == 'zscore':
                bin_means_norm = (bin_means - bin_means.mean()) / (bin_means.std() + 1e-10)
            elif normalization == 'none':
                bin_means_norm = bin_means
            else:
                raise ValueError(f"未知的Normalization方法: {normalization}")
            
            all_gene_curves[gene] = bin_means_norm
            
            # ========== 5. Plot curves ==========
            # Determine color and linestyle
            if config_color:
                color = config_color
            elif gene_colors and gene in gene_colors:
                color = gene_colors[gene]
            else:
                color = None
            
            if config_linestyle:
                linestyle = config_linestyle
            elif gene_linestyles and gene in gene_linestyles:
                linestyle = gene_linestyles[gene]
            else:
                linestyle = '-'
            
            # Plot curves
            if smooth > 0:
                # Create dense x values for smoothing
                x_smooth = np.linspace(bin_centers.min(), bin_centers.max(), config_n_bins * 10)
                
                try:
                    k = min(int(smooth), len(bin_centers) - 1, 5)
                    spl = make_interp_spline(bin_centers, bin_means_norm, k=k)
                    y_smooth = spl(x_smooth)
                    
         
                    if normalization == 'minmax':
                        y_smooth = np.clip(y_smooth, 0, 1)
                    
                    # Fill area under curve
                    if fill:
                        current_ax.fill_between(x_smooth, y_smooth, alpha=fill_alpha, color=color)
                    
                    # Draw smoothed curve
                    current_ax.plot(x_smooth, y_smooth, linestyle=linestyle, label=gene, 
                                   linewidth=linewidth, color=color, alpha=0.7)
                    
                    # Show original data points
                    if show_points:
                        current_ax.plot(bin_centers, bin_means_norm, 'o', markersize=4, 
                                       alpha=0.5, color=color)
                except Exception as e:
                    print(f"Warning: {gene} Smoothing failed ({str(e)})，Using 原始曲线")
                    marker = 'o-' if show_points else '-'
                    
                    if fill:
                        current_ax.fill_between(bin_centers, bin_means_norm, alpha=fill_alpha, color=color)
                    
                    current_ax.plot(bin_centers, bin_means_norm, marker, label=gene, 
                                   linewidth=linewidth, color=color,
                                   linestyle=linestyle if not show_points else '-',
                                   markersize=4 if show_points else 0, alpha=0.7)
            else:
                # Without smoothing
                marker = 'o-' if show_points else '-'
                
                if fill:
                    current_ax.fill_between(bin_centers, bin_means_norm, alpha=fill_alpha, color=color)
                
                current_ax.plot(bin_centers, bin_means_norm, marker, label=gene, 
                               linewidth=linewidth, color=color,
                               linestyle=linestyle if not show_points else '-',
                               markersize=4 if show_points else 0, alpha=0.7)
    
    # ========== 6. Add vertical reference lines ==========
    if vertical_lines:
        for line_pos in vertical_lines:
            ax.axvline(x=line_pos, color=vline_color, linestyle=vline_style,
                      linewidth=vline_width, alpha=vline_alpha, zorder=10)
    
    # ========== 7. Configure axes ==========

    if xlim:
        ax.set_xlim(xlim)
    if ylim:
        ax.set_ylim(ylim)
    if secondary_ylim and ax2:
        ax2.set_ylim(secondary_ylim)
    
    ax.set_xlabel(f'{coord_col} Position', fontsize=12)
    
    ylabel_dict = {
        'minmax': 'Normalized Expression (0-1)',
        'zscore': 'Z-score Normalized Expression',
        'none': 'Expression Level'
    }
    ax.set_ylabel(ylabel_dict.get(normalization, 'Expression'), fontsize=12)
    
    if ax2:
        ax2.set_ylabel(f'{ylabel_dict.get(normalization, "Expression")} (Secondary)', fontsize=12)
    
    ax.set_title(f'Gene Expression along {coord_col} axis', fontsize=14, fontweight='bold')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(x_tick_interval))
    
 
    if show_legend:
        lines1, labels1 = ax.get_legend_handles_labels()
        if ax2:
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, 
                     bbox_to_anchor=(1.15, 1), loc='upper left')
        else:
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    ax.margins(x=0, y=0.02)
    plt.tight_layout()
    
    return fig, all_gene_curves

In [ ]:
vertical_lines = [280,480,840, 2100,2600]
fig = plot_binned_density(
    nt, 
    n_bins=50,
    palette=sub_cluster_palette,
    figsize=(10, 5),
    smooth=2,  
    show_legend=False,
    linewidth=3,
    vertical_lines=vertical_lines,
    vline_width=3,
    density_method='count',
    coord_col='AP',
    group_col='leiden_hvg_sub',
    #secondary_y_groups=['e0_3', 'e0_4', 'e0_5'], 
    #dashed_groups=['e0_3', 'e0_4', 'e0_5'],
    #secondary_ylim=(0, 300),
    x_tick_interval=500,
    fill_alpha=0.5,
    #xlim=(-100,270),
    #ylim=(0,120),
    primary_y_groups=['e0_0', 'e0_1', 'e0_2', 'e0_3', 'e0_4', 'e0_5', 'e12_0', 'e12_1', 'e12_2', 'e12_3', 'e20_0', 'e20_1']
)
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure2\along_axix\density_plot_nt_leiden_hvg_sub_AP.pdf', dpi=900, bbox_inches='tight')
plt.show()

In [ ]:
vertical_lines = [280,480,840, 2100,2600]
gene_colors = {
    "FEZF1":  "#FF8C00",  # 0
    "EN1":  "#565DFD",  # 1
    "EGR2":  "#e5e022",  # 2
    "HOXB4": "#ff002a",  # 7
    "CDX2": "#66c102",  # 8
    "WNT3A": "#8e0119",  # 10
    #"DIO2": "#565DFD",  # 11
    #"KDR": "#ff002a",  # 14
   # "FZD10": "#AB76AE",  # 15
    #"CDH7": "#515151",  # 17
}


subset_values = nt.obs['leiden_hvg_sub'].unique().tolist()

fig, curves = plot_genes_along_AP(
    adata=nt,
    gene_configs=[
        {
            'genes': list(gene_colors.keys()),
            'subset_values': subset_values,  
            'use_secondary_y': False,
            'linestyle': '-',
            'n_bins': 50
        },
    ],
    coord_col='AP',
    subset_col='leiden_hvg_sub',  
    smooth=3,
    figsize=(8, 4),
    linewidth=3,
    fill=True,
    show_points=False,
    vertical_lines=vertical_lines,
    vline_width=3.0,
    show_legend=False,
    x_tick_interval=500,
    #secondary_ylim=(0, 1.2),
    gene_colors=gene_colors,
    fill_alpha=0.5,
    #xlim=(-200,2500),
    ylim=(0, 1.1)
)

#plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure2\along_axix\gene_nt_FEZF1_EN1_EGR2_HOXB4_CDX2_WNT3A_AP.pdf', dpi=900, bbox_inches='tight')
plt.show()

In [ ]:
vertical_lines = [280,480,840, 2100,2600]
fig = plot_binned_density(
    nt, 
    n_bins=50,
    palette=sub_cluster_palette,
    figsize=(10, 5),
    smooth=2,  # 轻微平滑
    show_legend=False,
    linewidth=3,
    vertical_lines=vertical_lines,
    vline_width=3,
    density_method='count',
    coord_col='AP',
    group_col='leiden_hvg_sub',
    #secondary_y_groups=['e0_3', 'e0_4', 'e0_5'], 
    #dashed_groups=['e0_3', 'e0_4', 'e0_5'],
    #secondary_ylim=(0, 300),
    x_tick_interval=500,
    fill_alpha=0.5,
    #xlim=(-100,270),
    ylim=(0,100),
    primary_y_groups=['e25_0', 'e25_1','e25_2']
)
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure2\along_axix\density_plot_nc_leiden_hvg_sub_AP.pdf', dpi=900, bbox_inches='tight')
plt.show()

# 6 Figure 2 g,h

In [ ]:
import squidpy as sq
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


adata =nt   
axis_name = 'AP'  
axis_col = 'AP'  
n_neighs = 15     
n_perms = 100     
n_jobs = 4       
fdr_threshold = 0.05  
output_prefix = 'squidpy_moranI_DV' 


print("=" * 60)
print(f"对{axis_name}
print("=" * 60)


print(f"{axis_name}")
spatial_key = f'spatial_{axis_name}'
adata.obsm[spatial_key] = np.column_stack([
    adata.obs[axis_col].values,  
    np.zeros(len(adata.obs))      
])


print(f"{axis_name}（n_neighs={n_neighs}）...")
sq.gr.spatial_neighbors(
    adata,
    coord_type='generic',
    n_neighs=n_neighs,
    spatial_key=spatial_key,
    key_added=spatial_key
)


print(f"{axis_name}Moran's I（n_perms={n_perms}）...")
sq.gr.spatial_autocorr(
    adata,
    mode='moran',
    n_perms=n_perms,
    n_jobs=n_jobs,
    connectivity_key=f'{spatial_key}_connectivities'
)


results = adata.uns['moranI'].copy()
results = results.sort_values('I', ascending=False)

print(f"\n {len(results)} Moran's I ({axis_name})")
print(f" (FDR<{fdr_threshold}): {(results['pval_norm_fdr_bh'] < fdr_threshold).sum()}")
print(f"\nTop 20 {axis_name}:")
print(results.head(20)[['I', 'pval_norm', 'pval_norm_fdr_bh']])


csv_file = f'{output_prefix}_results_gut.csv'
results.to_csv(csv_file)




fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, gene in enumerate(results.head(6).index):
    ax = axes[i]
    

    gene_expr = adata[:, gene].X.toarray().ravel() if hasattr(adata[:, gene].X, 'toarray') else adata[:, gene].X.ravel()
    axis_coords = adata.obs[axis_col].values
    
 
    scatter = ax.scatter(axis_coords, 
                        np.random.randn(len(axis_coords)) * 0.1,  
                        c=gene_expr, 
                        s=2, 
                        cmap='viridis', 
                        alpha=0.6)
    
 
    moran_i = results.loc[gene, 'I']
    pval = results.loc[gene, 'pval_norm_fdr_bh']
    ax.set_title(f'{gene}\nMoran I={moran_i:.3f}, FDR={pval:.2e}', fontsize=9)
    ax.set_xlabel(f'{axis_name} Position')
    ax.set_ylabel('Random jitter')
    plt.colorbar(scatter, ax=ax, label='Expression')

plt.tight_layout()
png_file = f'{output_prefix}_top6_genes.png'
plt.savefig(png_file, dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
genes = results[results['I'] > 0.1].index.tolist()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import binned_statistic, spearmanr
from scipy import sparse
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.ndimage import gaussian_filter1d
import warnings
warnings.filterwarnings('ignore')

def create_simple_comparison_heatmap(
    adata,
    gene_list,
    axis_col='AP',
    n_bins=100,
    figsize=(10, 10),
    gene_fontsize=9,
    axis_fontsize=12,
    title_fontsize=14,
    normalize_genes=True,
    cluster_genes=True,
    order_method='peak_grouped',  
    spearman_direction='desc',
    grouped_thr=0.2,
    binning='linear',
    smooth=None,
    smooth_sigma=1.2,
    rolling_window=5,
    min_cells_per_bin=None,
    save_path=None,
    cmap_raw='viridis',
    cmap_norm='RdBu_r',
    x_tick_interval=None,
    raw_layer_name='counts_raw',
    data_type='normalized',
    show_gene_labels=True,
    save_format='png',
    
    vmin=None, 
    vmax=None,                        
    vlines=None,                      
    vline_color='red',                
    vline_style='--',                 
    vline_width=1.5,                  
    vline_alpha=0.8,                  
    xlim=None,                        
    highlight_genes=None,             
    highlight_color='gold',           
    highlight_marker='◀',             
    highlight_size=12,                
    show_only_highlighted=False,      
    show_highlight_marker=True,       
    show_highlight_line=True,         
    highlight_label_position='left', 
    highlight_label_rotation=0,       
    show_ap_colorbar=False,           
    ap_colorbar_cmap='coolwarm',      
    ap_colorbar_height=0.02,

   
    hlines=None,             
    hline_color='white',     
    hline_style='-',         
    hline_width=2.0,          
    hline_alpha=1.0           
):
    

    
    if axis_col not in adata.obs.columns:
        raise ValueError(f"Axis column '{axis_col}' not found in adata.obs")
    missing = [g for g in gene_list if g not in adata.var_names]
    if missing:
        print(f"Warning: genes not found and will be dropped: {missing}")
    gene_list = [g for g in gene_list if g in adata.var_names]
    if not gene_list:
        raise ValueError("No valid genes in `gene_list`.")

    axis_vals = adata.obs[axis_col].values.astype(float)
    axis_vals = axis_vals[~np.isnan(axis_vals)]
    if len(axis_vals) == 0:
        raise ValueError(f"Column '{axis_col}' contains only NaNs.")
    axis_min, axis_max = np.nanmin(axis_vals), np.nanmax(axis_vals)

 
    if binning == 'quantile':
        qs = np.linspace(0, 1, n_bins + 1)
        bin_edges = np.quantile(axis_vals, qs)
        bin_edges[0], bin_edges[-1] = axis_min, axis_max
        unique_edges = []
        for i, edge in enumerate(bin_edges):
            if i == 0 or edge > unique_edges[-1]:
                unique_edges.append(edge)
            else:
                unique_edges.append(unique_edges[-1] + (axis_max - axis_min) * 1e-6)
        bin_edges = np.array(unique_edges)
        if len(np.unique(bin_edges)) < len(bin_edges):
            print(f"Warning: Quantile binning failed. Falling back to linear binning.")
            binning = 'linear'
    
    if binning == 'linear':
        bin_edges = np.linspace(axis_min, axis_max, n_bins + 1)
        eps = (axis_max - axis_min) * 1e-10
        for i in range(1, len(bin_edges)):
            if bin_edges[i] <= bin_edges[i-1]:
                bin_edges[i] = bin_edges[i-1] + eps

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    actual_bins = len(bin_edges) - 1
    print(f"Creating heatmap for {len(gene_list)} genes with {actual_bins} bins... (binning={binning}, data_type={data_type})")

  
    def build_heatmap_df(expr_matrix, label, min_cells_threshold=None):
        if sparse.issparse(expr_matrix):
            expr_matrix = expr_matrix.toarray()
        gene_idx = pd.Index(adata.var_names).get_indexer(gene_list)

        bin_counts = None
        if min_cells_threshold is not None:
            try:
                bin_counts, _, _ = binned_statistic(axis_vals, np.ones_like(axis_vals), statistic='count', bins=bin_edges)
            except ValueError as e:
                print(f"Warning: bin counting failed: {e}")
                bin_counts = None

        rows = []
        for idx in gene_idx:
            gexpr = expr_matrix[:, idx]
            try:
                means, _, _ = binned_statistic(axis_vals, gexpr, statistic='mean', bins=bin_edges)
            except ValueError as e:
                print(f"Warning: binned_statistic failed for gene {gene_list[gene_idx.tolist().index(idx)]}: {e}")
                means = np.zeros(len(bin_centers))

            if bin_counts is not None and min_cells_threshold is not None:
                means[bin_counts < min_cells_threshold] = np.nan

            if np.any(np.isnan(means)):
                ok = ~np.isnan(means)
                if ok.any():
                    means = np.interp(np.arange(len(means)), np.where(ok)[0], means[ok])
                else:
                    means = np.zeros_like(means)

            rows.append(means)

        df = pd.DataFrame(rows, index=gene_list, columns=[f'{v:.0f}' for v in bin_centers])
        return df

    raw_df = None
    norm_df = None
    
    if data_type in ['raw', 'both']:
        if raw_layer_name in adata.layers:
            raw_matrix = adata.layers[raw_layer_name]
        elif 'counts' in adata.layers:
            print(f"Note: '{raw_layer_name}' not found. Using 'counts' layer.")
            raw_matrix = adata.layers['counts']
        else:
            print(f"Note: '{raw_layer_name}' not found. Using adata.X as raw.")
            raw_matrix = adata.X
        raw_df = build_heatmap_df(raw_matrix, "Raw", min_cells_per_bin)
    
    if data_type in ['normalized', 'both']:
        norm_matrix = adata.X
        norm_df = build_heatmap_df(norm_matrix, "Normalized", min_cells_per_bin)

   
    def zscore_rows(df):
        arr = df.values
        mu = arr.mean(axis=1, keepdims=True)
        sd = arr.std(axis=1, keepdims=True)
        out = (arr - mu) / (sd + 1e-8)
        out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
        return pd.DataFrame(out, index=df.index, columns=df.columns)

    def prepare_plot_data(df, label):
        if normalize_genes:
            plot_df = zscore_rows(df)
            cbar_label = f"{label} (Z-score)"
            center = 0
        else:
            plot_df = df.copy()
            cbar_label = label
            center = None
        
       
        if smooth == 'gaussian':
            plot_df = pd.DataFrame(
                gaussian_filter1d(plot_df.values, sigma=smooth_sigma, axis=1, mode='nearest'),
                index=plot_df.index, columns=plot_df.columns
            )
        elif smooth == 'rolling':
            plot_df = (plot_df.T.rolling(rolling_window, center=True, min_periods=1).mean()).T
        
        return plot_df, cbar_label, center

    if data_type == 'raw':
        raw_plot, cbar_raw, center_val = prepare_plot_data(raw_df, "Raw")
        plot_dfs = [(raw_plot, cbar_raw, center_val, 'Raw', cmap_raw)]
        ref_df = raw_df
    elif data_type == 'normalized':
        norm_plot, cbar_norm, center_val = prepare_plot_data(norm_df, "Normalized")
        plot_dfs = [(norm_plot, cbar_norm, center_val, 'Normalized', cmap_norm)]
        ref_df = norm_df
    else:
        raw_plot, cbar_raw, center_raw = prepare_plot_data(raw_df, "Raw")
        norm_plot, cbar_norm, center_norm = prepare_plot_data(norm_df, "Normalized")
        plot_dfs = [
            (raw_plot, cbar_raw, center_raw, 'Raw', cmap_raw),
            (norm_plot, cbar_norm, center_norm, 'Normalized', cmap_norm)
        ]
        ref_df = norm_df


    def order_genes(method, df):
        M_plot = zscore_rows(df) if normalize_genes else df
        
     
        M_smooth_for_ordering = gaussian_filter1d(M_plot.values, sigma=2.0, axis=1)

        if method == 'none':
            return list(M_plot.index)

        if method == 'peak':
            peak_bin = M_smooth_for_ordering.argmax(axis=1) 
            gene_mean = M_plot.values.mean(axis=1)
            order = np.lexsort((-gene_mean, peak_bin))
            return M_plot.index[order].tolist()

        if method == 'spearman':
            x = np.arange(M_plot.shape[1])
            rho = np.array([spearmanr(x, M_plot.loc[g].values)[0] for g in M_plot.index])
            if spearman_direction == 'asc':
                order = np.argsort(rho)
            else:
                order = np.argsort(-rho)
            return M_plot.index[order].tolist()

        if method == 'grouped':
            x = np.arange(M_plot.shape[1])
            rho = np.array([spearmanr(x, M_plot.loc[g].values)[0] for g in M_plot.index])
            peak = M_smooth_for_ordering.argmax(axis=1) 

            left = rho < -grouped_thr
            mid = np.abs(rho) <= grouped_thr
            right = rho > grouped_thr

            left_idx = np.where(left)[0][np.argsort(peak[left])]
            mid_idx = np.where(mid)[0][np.argsort(peak[mid])]
            right_idx = np.where(right)[0][np.argsort(peak[right])]

            idx = np.r_[left_idx, mid_idx, right_idx]
            return M_plot.index[idx].tolist()

        if method == 'grouped_peak':
            x = np.arange(M_plot.shape[1])
            rho = np.array([spearmanr(x, M_plot.loc[g].values)[0] for g in M_plot.index])
            peak = M_smooth_for_ordering.argmax(axis=1) 
            mean_val = M_plot.values.mean(axis=1)

            group_vec = np.zeros_like(rho, dtype=int)
            group_vec[np.abs(rho) <= grouped_thr] = 1 
            group_vec[rho > grouped_thr] = 2          
            
            order = np.lexsort((-mean_val, peak, group_vec))
            return M_plot.index[order].tolist()

        if method == 'peak_grouped':
            x = np.arange(M_plot.shape[1])
            rho = np.array([spearmanr(x, M_plot.loc[g].values)[0] for g in M_plot.index])
            peak = M_smooth_for_ordering.argmax(axis=1) 
            mean_val = M_plot.values.mean(axis=1)

            group_id = np.zeros_like(rho, dtype=int)
            group_id[rho < -grouped_thr] = -1  
            group_id[rho > grouped_thr] = 1    
            
            order = np.lexsort((-mean_val, group_id, peak))
            return M_plot.index[order].tolist()

        M = df.values
        Mz = (M - M.mean(axis=1, keepdims=True)) / (M.std(axis=1, keepdims=True) + 1e-12)
        Mz = np.nan_to_num(Mz, nan=0.0, posinf=0.0, neginf=0.0)
        row_std = Mz.std(axis=1)
        keep = np.isfinite(Mz).all(axis=1) & (row_std > 1e-8)
        kept_genes = np.array(gene_list)[keep]
        if len(kept_genes) >= 2:
            Z = linkage(Mz[keep, :], method='ward', metric='euclidean')
            leaves = leaves_list(Z)
            ordered = kept_genes[leaves].tolist()
            dropped = [g for g in M_plot.index if g not in ordered]
            ordered += dropped
            return ordered
        else:
            return list(M_plot.index)

    valid_methods = {'cluster', 'peak', 'spearman', 'grouped', 'grouped_peak', 'peak_grouped', 'none'}
    method = order_method if order_method in valid_methods else ('cluster' if cluster_genes else 'none')
    
    gene_order = order_genes(method, ref_df)

    plot_dfs = [(df.reindex(gene_order), cbar, center, title, cmap) for df, cbar, center, title, cmap in plot_dfs]


    n_plots = len(plot_dfs)
    if n_plots == 1:
        fig, ax = plt.subplots(1, 1, figsize=figsize)
        axes = [ax]
    else:
        fig, axes = plt.subplots(1, 2, figsize=(figsize[0]*2, figsize[1]))

    if x_tick_interval is None:
        if actual_bins <= 20: x_tick_interval = 2
        elif actual_bins <= 50: x_tick_interval = 5
        elif actual_bins <= 100: x_tick_interval = 10
        elif actual_bins <= 150: x_tick_interval = 15
        else: x_tick_interval = 20

    for idx, (plot_df, cbar_label, center, title, cmap) in enumerate(plot_dfs):
        ax = axes[idx]
        
        if show_only_highlighted and highlight_genes is not None:
            if highlight_label_position == 'right':
                ytick_labels = False
                show_yticks = False
            else:
                ytick_labels = [gene if gene in highlight_genes else '' for gene in plot_df.index]
                show_yticks = True
        elif show_gene_labels:
            ytick_labels = True
            show_yticks = True
        else:
            ytick_labels = False
            show_yticks = False
        
        sns.heatmap(
            plot_df, ax=ax, cmap=cmap, center=center, 
            vmin=vmin, vmax=vmax,
            cbar=True,
            cbar_kws={'label': cbar_label, 'shrink': 0.8},
            xticklabels=x_tick_interval, 
            yticklabels=ytick_labels,
            linewidths=0
        )
        
    
        if hlines is not None:
            for h_gene in hlines:
                if h_gene in plot_df.index:
                  
                    gene_idx = plot_df.index.get_loc(h_gene)
                    
                    line_pos = gene_idx + 1
                    ax.axhline(
                        y=line_pos, 
                        color=hline_color, 
                        linestyle=hline_style, 
                        linewidth=hline_width, 
                        alpha=hline_alpha
                    )

   
        if show_only_highlighted and highlight_genes is not None:
            if highlight_label_position == 'right':
                ax.tick_params(axis='y', which='both', left=False, labelleft=False)
            else:
                ax.tick_params(axis='y', which='both', length=0)
        
        ax.set_title(title, fontsize=title_fontsize, fontweight='bold')
        ax.set_ylabel('Genes' if (show_gene_labels or (show_only_highlighted and highlight_label_position=='left')) else '', 
                      fontsize=axis_fontsize, fontweight='bold')
        ax.set_xlabel(f'{axis_col} (binned)', fontsize=axis_fontsize, fontweight='bold')
        
        if show_yticks:
            ax.tick_params(axis='y', labelsize=gene_fontsize, rotation=highlight_label_rotation)
        ax.tick_params(axis='x', labelsize=axis_fontsize-1, rotation=45)

        if vlines is not None:
            for vline_pos in vlines:
                bin_idx = np.searchsorted(bin_centers, vline_pos)
                if 0 <= bin_idx < len(bin_centers):
                    ax.axvline(
                        x=bin_idx + 0.5, color=vline_color, linestyle=vline_style, 
                        linewidth=vline_width, alpha=vline_alpha, zorder=10
                    )
        
        if xlim is not None:
            actual_xlim = (max(xlim[0], axis_min), min(xlim[1], axis_max))
            x_start = np.searchsorted(bin_centers, actual_xlim[0])
            x_end = np.searchsorted(bin_centers, actual_xlim[1])
            x_start = max(0, x_start)
            x_end = min(len(bin_centers), x_end)
            if x_start < x_end:
                ax.set_xlim(x_start, x_end)
            else:
                print(f"Warning: xlim {xlim} results in invalid range. Using full range.")
        
        if highlight_genes is not None:
            for gene in highlight_genes:
                if gene in plot_df.index:
                    gene_idx = plot_df.index.get_loc(gene)
                    y_pos = gene_idx + 0.5
                    if highlight_label_position == 'right':
                        ax.text(plot_df.shape[1] + 0.5, y_pos, gene, fontsize=gene_fontsize + 2,
                                color=highlight_color, ha='left', va='center', fontweight='bold',
                                rotation=highlight_label_rotation, clip_on=False, zorder=100)
                        if show_highlight_line:
                            ax.plot([plot_df.shape[1], plot_df.shape[1] + 0.3], [y_pos, y_pos],
                                    color=highlight_color, linewidth=1.5, solid_capstyle='round',
                                    clip_on=False, zorder=100)
                    else:
                        if show_highlight_line:
                            ax.plot([-0.5, 0], [y_pos, y_pos], color=highlight_color, linewidth=1.5,
                                    solid_capstyle='round', clip_on=False, zorder=100)
                        if show_highlight_marker:
                            ax.plot([-1.5, -0.5], [y_pos, y_pos], color=highlight_color, linewidth=2,
                                    solid_capstyle='round', clip_on=False, zorder=100)
                            ax.text(-1.8, y_pos, highlight_marker, fontsize=highlight_size, 
                                    color=highlight_color, ha='right', va='center', fontweight='bold',
                                    clip_on=False, zorder=100)
                        if show_yticks:
                            yticks = ax.get_yticklabels()
                            for label in yticks:
                                if label.get_text() == gene:
                                    label.set_color(highlight_color)
                                    label.set_fontweight('bold')
                                    label.set_fontsize(gene_fontsize + 2)
                                    label.set_rotation(highlight_label_rotation)
        
        if show_ap_colorbar:
            pos = ax.get_position()
            cbar_ax_list = [a for a in fig.get_axes() if a != ax and a.get_label() != 'ap_colorbar']
            heatmap_right = pos.x1
            for other_ax in cbar_ax_list:
                other_pos = other_ax.get_position()
                if other_pos.x0 >= pos.x0 and other_pos.y0 < pos.y1 and other_pos.y1 > pos.y0:
                    heatmap_right = min(heatmap_right, other_pos.x0 - 0.01)
            ap_cbar_ax = fig.add_axes([pos.x0, pos.y1 + 0.01, heatmap_right - pos.x0, ap_colorbar_height])
            ap_cbar_ax.set_label('ap_colorbar')
            gradient = np.linspace(0, 1, len(bin_centers)).reshape(1, -1)
            ap_cbar_ax.imshow(gradient, aspect='auto', cmap=ap_colorbar_cmap, extent=[0, len(bin_centers), 0, 1])
            ap_cbar_ax.set_xticks([]); ap_cbar_ax.set_yticks([])
            ap_cbar_ax.set_ylabel(f'{axis_col}', fontsize=axis_fontsize-2, rotation=0, ha='right', va='center')
            ap_cbar_ax.text(-0.5, 0.5, f'{axis_min:.0f}', fontsize=axis_fontsize-2, ha='right', va='center')
            ap_cbar_ax.text(len(bin_centers)+0.5, 0.5, f'{axis_max:.0f}', fontsize=axis_fontsize-2, ha='left', va='center')
            for spine in ap_cbar_ax.spines.values(): spine.set_visible(False)

    if n_plots == 1:
        title_text = f'Gene Expression: {data_type.capitalize()}\n({len(gene_list)} genes, {actual_bins} bins along {axis_col}; order={method})'
    else:
        title_text = f'Comparison: Raw vs Normalized\n({len(gene_list)} genes, {actual_bins} bins; order={method})'
    
    fig.suptitle(title_text, fontsize=title_fontsize+2, fontweight='bold', y=0.98 if n_plots == 1 else 0.95)
    
    param_text = f'Binning={binning}, Smooth={smooth}, Z-score={normalize_genes}'
    if show_only_highlighted: param_text += ', Showing only highlighted gene labels'
    else: param_text += f', Gene labels={show_gene_labels}'
    if vmin is not None or vmax is not None: param_text += f', vmin={vmin}, vmax={vmax}'
    if vlines is not None: param_text += f', vlines={len(vlines)} lines'
    if highlight_genes is not None: param_text += f', highlighted={len(highlight_genes)} genes'
    if hlines is not None: param_text += f', hlines={len(hlines)} lines'
    
    fig.text(0.02, 0.02, param_text, fontsize=8, alpha=0.7)

    plt.tight_layout()
    
    if save_path:
        import os
        save_dir = os.path.dirname(save_path)
        if save_dir and not os.path.exists(save_dir):
            os.makedirs(save_dir, exist_ok=True)
        if save_path.endswith('.pdf'): fmt = 'pdf'
        elif save_path.endswith('.png'): fmt = 'png'
        elif save_path.endswith('.svg'): fmt = 'svg'
        else: fmt = save_format; save_path = f"{save_path}.{fmt}"
        
        try:
            plt.savefig(save_path, dpi=300, bbox_inches='tight', format=fmt)
            print(f"✓ Saved to: {save_path}")
        except Exception as e:
            print(f"✗ Error saving file: {e}")
    
    plt.show()

    print("Done.")
    print(f"  Genes: {len(gene_list)} | Bins: {actual_bins} | Range: {axis_min:.3f}-{axis_max:.3f}")
    print(f"  Order method: {method} | Data type: {data_type}")
    if highlight_genes:
        print(f"  Highlighted genes: {highlight_genes}")

    if data_type == 'raw': return raw_df
    elif data_type == 'normalized': return norm_df
    else: return raw_df, norm_df

In [ ]:
 norm_df = create_simple_comparison_heatmap(
   nt, genes, axis_col='AP',
    n_bins=40, figsize=(6, 6),
    order_method='peak_grouped',  #spearman   peak
    binning='linear',#quantile   linear
    smooth='gaussian', smooth_sigma=2,
    normalize_genes=True,
    data_type='normalized',
    show_only_highlighted=True,      
    #show_highlight_marker=False, 
    vmin=-2, vmax=2,
    vlines=[280,480,840, 2100,2600],
    vline_color='blue',
    vline_style='--',
    hlines=["NKX2-1","PAX8","PCDH8","COL2A1","CDX1"],
    hline_color='black',
    hline_width=2,
    #xlim=(0, 2500),
    highlight_genes=["FEZF1",'EN1','EGR2',"IRX1","CDX2","WNT3A"],
    highlight_color='red',
    highlight_size=2,
    highlight_label_position='left',
    save_path="P:/PI/PI_Chuva_de_Sousa_Lopes/susana/Fu/Project/20_Xenium 5k embryo/paper/3st_2025_1_27/Figure2/along_axix/NT_AP_axix_gene_heatmap_group_peaked_group_1st.pdf"
     
)

In [ ]:
import squidpy as sq
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


adata =nt   
axis_name = 'DV'  
axis_col = 'DV'  
n_neighs = 15     
n_perms = 100     
n_jobs = 4       
fdr_threshold = 0.05  
output_prefix = 'squidpy_moranI_DV' 


print("=" * 60)
print(f"对{axis_name}
print("=" * 60)


print(f"{axis_name}")
spatial_key = f'spatial_{axis_name}'
adata.obsm[spatial_key] = np.column_stack([
    adata.obs[axis_col].values,  
    np.zeros(len(adata.obs))      
])


print(f"{axis_name}（n_neighs={n_neighs}）...")
sq.gr.spatial_neighbors(
    adata,
    coord_type='generic',
    n_neighs=n_neighs,
    spatial_key=spatial_key,
    key_added=spatial_key
)


print(f"{axis_name}Moran's I（n_perms={n_perms}）...")
sq.gr.spatial_autocorr(
    adata,
    mode='moran',
    n_perms=n_perms,
    n_jobs=n_jobs,
    connectivity_key=f'{spatial_key}_connectivities'
)


results = adata.uns['moranI'].copy()
results = results.sort_values('I', ascending=False)

print(f"\n {len(results)} Moran's I ({axis_name})")
print(f" (FDR<{fdr_threshold}): {(results['pval_norm_fdr_bh'] < fdr_threshold).sum()}")
print(f"\nTop 20 {axis_name}:")
print(results.head(20)[['I', 'pval_norm', 'pval_norm_fdr_bh']])


csv_file = f'{output_prefix}_results_gut.csv'
results.to_csv(csv_file)




fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, gene in enumerate(results.head(6).index):
    ax = axes[i]
    

    gene_expr = adata[:, gene].X.toarray().ravel() if hasattr(adata[:, gene].X, 'toarray') else adata[:, gene].X.ravel()
    axis_coords = adata.obs[axis_col].values
    
 
    scatter = ax.scatter(axis_coords, 
                        np.random.randn(len(axis_coords)) * 0.1,  
                        c=gene_expr, 
                        s=2, 
                        cmap='viridis', 
                        alpha=0.6)
    
 
    moran_i = results.loc[gene, 'I']
    pval = results.loc[gene, 'pval_norm_fdr_bh']
    ax.set_title(f'{gene}\nMoran I={moran_i:.3f}, FDR={pval:.2e}', fontsize=9)
    ax.set_xlabel(f'{axis_name} Position')
    ax.set_ylabel('Random jitter')
    plt.colorbar(scatter, ax=ax, label='Expression')

plt.tight_layout()
png_file = f'{output_prefix}_top6_genes.png'
plt.savefig(png_file, dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
genes = results[results['I'] > 0.1].index.tolist()

In [ ]:
 norm_df = create_simple_comparison_heatmap(
   data, genes, axis_col='DV',
    n_bins=30, figsize=(5, 6),
    order_method='peak_grouped',  #spearman   peak
    binning='linear',#quantile   linear
    smooth='gaussian', smooth_sigma=2,
    normalize_genes=True,
    data_type='normalized',
    show_only_highlighted=True,      # 只显示HOXB9和RELN
    #show_highlight_marker=False, 
    vmin=-2, vmax=2,
    #vlines=[70,125,200],
    vline_color='blue',
    vline_style='--',
    hlines=["FOXA2","OLIG2","PAX6"],
    hline_color='black',
    hline_width=2,
    #xlim=(0, 280),
    highlight_genes=["FOXA2",'PAX6',"WNT3A"],
    highlight_color='red',
    highlight_size=2,
    highlight_label_position='left',
    save_path="P:/PI/PI_Chuva_de_Sousa_Lopes/susana/Fu/Project/20_Xenium 5k embryo/paper/3st_2025_1_27/Figure2/along_axix/NT_DV_axix_gene_heatmap_group_s3_s4_1st.pdf"
     
)

#  7,Figure 2j

In [ ]:
df= pd.read_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure2\DEG\cellnest_df.xlsx", index=True)

In [ ]:
def create_detailed_interaction_adata(df, axis_col='AP', score_col='specificity_score', n_bins=150):

    import anndata as ad
    import pandas as pd
    import numpy as np

    df = df.copy()
    

    df['interaction_id'] = (
        df['ligand'].astype(str) + "_" + df['receptor'].astype(str) + "__" + 
        df['leiden_from'].astype(str) + "__" + 
        df['leiden_to'].astype(str)
    )
    
   
    min_val, max_val = df[axis_col].min(), df[axis_col].max()
    
    bin_edges = np.linspace(min_val, max_val, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    bin_indices = np.searchsorted(bin_edges, df[axis_col].values) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)
    df['bin_id'] = bin_indices

 
    matrix_df = df.pivot_table(
        index='bin_id', 
        columns='interaction_id', 
        values=score_col, 
        aggfunc='sum',
        fill_value=0
    )
    

    full_bins = pd.RangeIndex(start=0, stop=n_bins, name='bin_id')
    matrix_df = matrix_df.reindex(full_bins, fill_value=0)
    

    adata = ad.AnnData(X=matrix_df.values)
    

    adata.var_names = matrix_df.columns
    
 
    meta_df = pd.Series(matrix_df.columns).str.split('__', expand=True)
    

    adata.var['ligand_receptor'] = meta_df[0].values
    adata.var['sender'] = meta_df[1].values
    adata.var['receiver'] = meta_df[2].values
    
    adata.var['interaction_type'] = (
        adata.var['ligand_receptor'] + " (" + 
        adata.var['sender'] + "->" + 
        adata.var['receiver'] + ")"
    )
    

    adata.obs_names = [f"Bin_{i}" for i in matrix_df.index]
    adata.obs['bin_id'] = matrix_df.index
    adata.obs[axis_col] = bin_centers
    

    spatial_coords = np.column_stack([bin_centers, np.zeros_like(bin_centers)])
    adata.obsm['spatial'] = spatial_coords
    adata.obsm[f'spatial_{axis_col}'] = spatial_coords
    

    return adata


adata_detailed = create_detailed_interaction_adata(
    df, 
    axis_col='AP', 
    score_col='specificity_score', 
    n_bins=600
)


In [ ]:
import squidpy as sq
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


adata =adata_detailed   
axis_name = 'AP'  
axis_col = 'AP' 
n_neighs = 6     
n_perms = 100    
n_jobs = 4        
fdr_threshold = 0.05  
output_prefix = 'squidpy_moranI_DV' 



spatial_key = f'spatial_{axis_name}'
adata.obsm[spatial_key] = np.column_stack([
    adata.obs[axis_col].values,  
    np.zeros(len(adata.obs))    
])


sq.gr.spatial_neighbors(
    adata,
    coord_type='generic',
    n_neighs=n_neighs,
    spatial_key=spatial_key,
    key_added=spatial_key
)


sq.gr.spatial_autocorr(
    adata,
    mode='moran',
    n_perms=n_perms,
    n_jobs=n_jobs,
    connectivity_key=f'{spatial_key}_connectivities'
)

results = adata.uns['moranI'].copy()
results = results.sort_values('I', ascending=False)


fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, gene in enumerate(results.head(6).index):
    ax = axes[i]
    
   
    gene_expr = adata[:, gene].X.toarray().ravel() if hasattr(adata[:, gene].X, 'toarray') else adata[:, gene].X.ravel()
    axis_coords = adata.obs[axis_col].values
    
 
    scatter = ax.scatter(axis_coords, 
                        np.random.randn(len(axis_coords)) * 0.1,  
                        c=gene_expr, 
                        s=2, 
                        cmap='viridis', 
                        alpha=0.6)
    
    
    moran_i = results.loc[gene, 'I']
    pval = results.loc[gene, 'pval_norm_fdr_bh']
    ax.set_title(f'{gene}\nMoran I={moran_i:.3f}, FDR={pval:.2e}', fontsize=9)
    ax.set_xlabel(f'{axis_name} Position')
    ax.set_ylabel('Random jitter')
    plt.colorbar(scatter, ax=ax, label='Expression')

plt.tight_layout()
png_file = f'{output_prefix}_top6_genes.png'
plt.savefig(png_file, dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy.stats import binned_statistic, spearmanr
from scipy import sparse
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.ndimage import gaussian_filter1d
import warnings
warnings.filterwarnings('ignore')

def create_advanced_lr_heatmap(
    adata,
    gene_list,
    axis_col='AP',
    sender_col='sender',     
    receiver_col='receiver',
    cluster_colors=None,   
    n_bins=100,
    figsize=(12, 16),        
    gene_fontsize=9,
    axis_fontsize=12,
    title_fontsize=14,
    normalize_genes=True,
    order_method='peak_grouped', 
    spearman_direction='desc',
    grouped_thr=0.2,
    binning='linear',
    smooth=None,
    smooth_sigma=1.2,
    rolling_window=5,
    min_cells_per_bin=None,
    save_path=None,
    cmap_raw='viridis',
    cmap_norm='RdBu_r',
    x_tick_interval=None,
    data_type='normalized', w
    show_gene_labels=True,
    save_format='png',
    
    vmin=None, vmax=None,
    vlines=None, vline_color='blue', vline_style='--', vline_width=1.5, vline_alpha=0.8,
    xlim=None,
    highlight_genes=None, highlight_color='red', highlight_marker='◀', highlight_size=12,
    show_only_highlighted=False, show_highlight_marker=True, show_highlight_line=True,
    highlight_label_position='left', highlight_label_rotation=0,
    show_ap_colorbar=False, ap_colorbar_cmap='coolwarm', ap_colorbar_height=0.02
):



    missing = [g for g in gene_list if g not in adata.var_names]
    if missing:
        print(f"Warning: genes not found and will be dropped: {missing}")
    gene_list = [g for g in gene_list if g in adata.var_names]
    if not gene_list:
        raise ValueError("No valid genes in `gene_list`.")

    
    if axis_col not in adata.obs.columns:
        raise ValueError(f"Axis column '{axis_col}' not found in adata.obs")
    
    axis_vals = adata.obs[axis_col].values.astype(float)
  
    mask = ~np.isnan(axis_vals)
    axis_vals = axis_vals[mask]
    
    if len(axis_vals) == 0:
        raise ValueError(f"Column '{axis_col}' contains only NaNs.")
    axis_min, axis_max = np.min(axis_vals), np.max(axis_vals)

    
    if binning == 'linear':
        bin_edges = np.linspace(axis_min, axis_max, n_bins + 1)
        
        bin_edges[-1] += 1e-6 
    elif binning == 'quantile':
        bin_edges = np.quantile(axis_vals, np.linspace(0, 1, n_bins + 1))
        
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    actual_bins = len(bin_centers)

   
    if sparse.issparse(adata.X):
        X_subset = adata[:, gene_list].X.toarray()
    else:
        X_subset = adata[:, gene_list].X
    
    
    if len(mask) != X_subset.shape[0]:
         X_subset = X_subset[mask, :]

    
    from scipy.stats import binned_statistic
    
   
    bin_means, _, _ = binned_statistic(
        axis_vals, 
        X_subset.T, 
        statistic='mean', 
        bins=bin_edges
    )
    
   
    df = pd.DataFrame(bin_means, index=gene_list, columns=[f'{v:.0f}' for v in bin_centers])
    
  
    df = df.fillna(0)
    
    
    def zscore_rows(d):
        arr = d.values
        mu = arr.mean(axis=1, keepdims=True)
        sd = arr.std(axis=1, keepdims=True)
        out = (arr - mu) / (sd + 1e-8)
        return pd.DataFrame(out, index=d.index, columns=d.columns)

    if normalize_genes:
        plot_df = zscore_rows(df)
        cbar_label = "Z-score"
        center = 0
    else:
        plot_df = df.copy()
        cbar_label = "Expression"
        center = None

    if smooth == 'gaussian':
        plot_df = pd.DataFrame(
            gaussian_filter1d(plot_df.values, sigma=smooth_sigma, axis=1, mode='nearest'),
            index=plot_df.index, columns=plot_df.columns
        )

  
    M_plot = plot_df
   
    M_smooth = gaussian_filter1d(M_plot.values, sigma=2.0, axis=1)
    
    if order_method == 'peak_grouped':
        peak = M_smooth.argmax(axis=1)
        mean_val = M_plot.values.mean(axis=1)
        x = np.arange(M_plot.shape[1])
        rho = np.array([spearmanr(x, row)[0] for row in M_plot.values])
        
        group_id = np.zeros_like(rho, dtype=int)
        group_id[rho < -grouped_thr] = -1 # Down
        group_id[rho > grouped_thr] = 1   # Up
        
        order = np.lexsort((-mean_val, group_id, peak))
        gene_order = M_plot.index[order].tolist()
    else:
       
        peak = M_smooth.argmax(axis=1)
        order = np.argsort(peak)
        gene_order = M_plot.index[order].tolist()

    
    plot_df = plot_df.reindex(gene_order)

   
    meta_df = adata.var.loc[gene_order, [sender_col, receiver_col]]
    
    
    unique_clusters = set(meta_df[sender_col].unique()) | set(meta_df[receiver_col].unique())
    if cluster_colors is None:
        
        palette = sns.color_palette('tab20', n_colors=len(unique_clusters))
        cluster_colors = dict(zip(sorted(list(unique_clusters)), palette))
    
   
    sender_colors = meta_df[sender_col].map(cluster_colors).tolist()
    receiver_colors = meta_df[receiver_col].map(cluster_colors).tolist()
    
    
    import matplotlib.colors as mcolors
    sender_rgb = np.array([mcolors.to_rgb(c) for c in sender_colors]).reshape(-1, 1, 3)
    receiver_rgb = np.array([mcolors.to_rgb(c) for c in receiver_colors]).reshape(-1, 1, 3)

   
    fig = plt.figure(figsize=figsize)
    
   
    gs = gridspec.GridSpec(1, 4, width_ratios=[0.03, 0.03, 0.01, 1], wspace=0.05)
    
    ax_sender = fig.add_subplot(gs[0])
    ax_receiver = fig.add_subplot(gs[1])
    
    ax_heatmap = fig.add_subplot(gs[3])

   
    ax_sender.imshow(sender_rgb, aspect='auto', interpolation='nearest')
    ax_sender.set_title('From', fontsize=10, rotation=45)
    ax_sender.axis('off')
    
    ax_receiver.imshow(receiver_rgb, aspect='auto', interpolation='nearest')
    ax_receiver.set_title('To', fontsize=10, rotation=45)
    ax_receiver.axis('off')

    
    if show_only_highlighted and highlight_genes is not None:
        if highlight_label_position == 'right':
            ytick_labels = False #
        else:
           
            ytick_labels = [g if g in highlight_genes else '' for g in plot_df.index]
    elif show_gene_labels:
        ytick_labels = True 
    else:
        ytick_labels = False

    sns.heatmap(
        plot_df, 
        ax=ax_heatmap, 
        cmap=cmap_norm, 
        center=center,
        vmin=vmin, vmax=vmax,
        cbar=True,
        cbar_kws={'label': cbar_label, 'shrink': 0.5},
        xticklabels=x_tick_interval if x_tick_interval else 10,
        yticklabels=ytick_labels
    )

  
    ax_heatmap.set_xlabel(f'{axis_col} (binned)', fontsize=axis_fontsize, fontweight='bold')
    ax_heatmap.set_ylabel('') 

    plt.setp(ax_heatmap.get_xticklabels(), rotation=45, ha='right')
    
  
    if show_gene_labels or (show_only_highlighted and highlight_label_position != 'right'):
         ax_heatmap.tick_params(axis='y', labelsize=gene_fontsize)


    if vlines is not None:
        for vline_pos in vlines:
            bin_idx = np.searchsorted(bin_centers, vline_pos)
            if 0 <= bin_idx < len(bin_centers):
                ax_heatmap.axvline(x=bin_idx + 0.5, color=vline_color, linestyle=vline_style, 
                                   linewidth=vline_width, alpha=vline_alpha)

    
    if highlight_genes is not None:
        for gene in highlight_genes:
            if gene in plot_df.index:
                
                y_pos = plot_df.index.get_loc(gene) + 0.5
                
                if highlight_label_position == 'right':
                   
                    ax_heatmap.text(
                        plot_df.shape[1] + 0.5, y_pos, 
                        gene, 
                        fontsize=gene_fontsize + 2,
                        color=highlight_color, 
                        ha='left', va='center', fontweight='bold'
                    )
                    if show_highlight_line:
                        ax_heatmap.plot(
                            [plot_df.shape[1], plot_df.shape[1] + 0.3], [y_pos, y_pos],
                            color=highlight_color, linewidth=1, clip_on=False
                        )
                else:
                  
                    try:
                         
                        idx = plot_df.index.get_loc(gene)
                        if idx < len(yticks):
                            yticks[idx].set_color(highlight_color)
                            yticks[idx].set_fontweight('bold')
                    except:
                        pass

  
    used_clusters = set(meta_df[sender_col]) | set(meta_df[receiver_col])
    legend_patches = [mpatches.Patch(color=cluster_colors[c], label=c) for c in sorted(list(used_clusters))]
    
    
    fig.legend(
        handles=legend_patches, 
        title="Cell Clusters",
        bbox_to_anchor=(1.02, 0.8), 
        loc='upper left',
        borderaxespad=0.
    )
    
    plt.tight_layout()
    
 
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✓ Saved to: {save_path}")

    return plot_df

In [ ]:

moran_df = adata_detailed.uns['moranI'].copy()
is_significant = moran_df['pval_norm_fdr_bh'] < 0.05


is_paracrine = moran_df['sender'] != moran_df['receiver']


filtered_results = moran_df[is_significant & is_paracrine & is_positive].copy()


genes = filtered_results.index.tolist()


In [ ]:

norm_df = create_advanced_lr_heatmap(
    adata_detailed, 
    gene_list=genes, 
    axis_col='AP',
    sender_col='sender',     
    receiver_col='receiver',
    cluster_colors=sub_cluster_palette, 
    n_bins=100,
    figsize=(10, 14),       
    order_method='peak_grouped',
    vlines=[280,480,840, 2100,2600],
    smooth='gaussian', smooth_sigma=1.5,
    highlight_genes=['WNT5A_FZD10__e20_1__e20_0'], 
    highlight_label_position='right',
    vmin=-2, vmax=2,
    show_only_highlighted=True,
   cluster_colors=sub_cluster_palette
    
)